# MoneyPrint V4 — Laboratório Colab (ArtiSys)

Execução interativa de **roteiro → storyboard canônico V3 → voz PT-BR → sincronismo em frames → renderização incremental → QA → Shorts**.

**Core R$ 0 / self-hosted / open source.** Google Colab é somente um executor experimental, não é backend permanente nem serviço obrigatório. Na modalidade gratuita, CPU/GPU e duração das sessões não são garantidas. Nada é publicado automaticamente.

Este notebook utiliza o repositório **nutricionistaalmeidavh-spec/moneyPrintTest** numa branch de desenvolvimento. Para produção, abra a versão mergeada na main após testes. O pipeline aproveita o editor FFmpeg Studio V3, Paradee e faster-whisper existentes; não gera notícias ou mídia sem proveniência.

**Avisos:** os testes automatizados não avaliam naturalidade da locução ou estética das cenas. Revise o MP4 antes de publicar. Qwen3 requer GPU e uma referência de voz que você tem direito de utilizar. Remotion é opcional, source-available e sujeito a licença própria.


## 1. Preparar runtime e dependências gratuitas

No menu **Ambiente de execução → Alterar tipo de ambiente**, selecione GPU T4 somente se disponível e desejada. A voz padrão Paradee funciona também em CPU.

Esta etapa instala pacotes open source no ambiente temporário da sessão. Não instala Ollama, não utiliza APIs pagas e não altera tokens/secrets.


In [ ]:
import os, sys, subprocess, time, json, shutil
from pathlib import Path
print("Python:", sys.version.split()[0])
print("GPU:", subprocess.getoutput("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader") if shutil.which("nvidia-smi") else "não disponível; modo CPU")
!apt-get update -qq >/dev/null
!apt-get install -y -qq ffmpeg espeak-ng libespeak-ng1 fonts-lato fonts-dejavu-core >/dev/null
!pip install -q 'numpy>=1.26,<3' 'onnxruntime>=1.19,<2' 'soundfile>=0.12,<1' 'scipy>=1.11' 'misaki[pt]' 'espeakng-loader' 'phonemizer>=3.3,<4' 'huggingface-hub>=0.25,<2' 'faster-whisper>=1.1,<2' 'pillow>=10,<12'
REPO = Path("/content/moneyPrintTest")
REF = "feat/moneyprint-v4-colab-lab"  # após merge, mudar para main
if not REPO.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REF,
                    "https://github.com/nutricionistaalmeidavh-spec/moneyPrintTest.git",
                    str(REPO)], check=True)
os.chdir(REPO)
sys.path.insert(0, str(REPO))
print("Revisão:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())
print("FFmpeg:", subprocess.getoutput("ffmpeg -version").splitlines()[0])


## 2. QA sem GPU: executar testes antes da produção

O teste valida a canonicidade da timeline e a reutilização de cenas. Falhas interrompem o fluxo.


In [ ]:
subprocess.run([sys.executable, "-m", "unittest", "tests.test_colab_v4", "-v"], check=True, cwd=REPO)


## 3. Configurar o vídeo

Digite o roteiro completo abaixo. Frases e parágrafos viram cenas editáveis. A duração final é **calculada pelo áudio realmente gerado**, não por estimativas.

`VOICE_ENGINE` suporta `paradee` em CPU (padrão) ou `qwen3` na GPU (experimental). Qwen3 exige referência de voz autorizada e transcrição exata.

Mantenha `PERSISTIR_NO_DRIVE=False` para evitar autorizar acesso ao Google Drive. Se habilitar, o notebook solicitará a montagem explicitamente.


In [ ]:
TOPIC = "Como funciona uma análise de pentest com ferramentas gratuitas"
SCRIPT = """Uma análise de pentest começa definindo o escopo e obtendo autorização.
Depois, ferramentas como Nmap, Nuclei e OWASP ZAP ajudam a identificar possíveis vulnerabilidades.
O resultado não é apenas uma lista de alertas: é uma avaliação com evidências, riscos e correções recomendadas.
Com ferramentas de código aberto, é possível montar um laboratório de análise sem depender de serviços pagos."""

ASPECT = "16:9"             # "16:9", "9:16", "1:1"
VISUAL_STYLE = "light_motion"  # "light_motion" ou "classic"; ambos gratuitos
VOICE_ENGINE = "paradee"    # "paradee" (CPU) ou "qwen3" (GPU opcional)
VOICE_SPEED = 1.06          # 0.80 a 1.25; Qwen ignora ajuste de velocidade
QWEN_REFERENCE_AUDIO = ""   # caminho para WAV autorizado, obrigatório quando qwen3
QWEN_REFERENCE_TEXT = ""    # transcrição exata do WAV
PERSISTIR_NO_DRIVE = False # persistência opcional com autorização explícita
GENERATE_WORD_TIMESTAMPS = True
GENERATE_SHORTS = False     # habilite para vídeos com >=20 segundos de fala contínua
SHORTS_COUNT = 5

if PERSISTIR_NO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = Path("/content/drive/MyDrive/MoneyPrintV4")
else:
    WORK = Path("/content/MoneyPrintV4")
WORK.mkdir(parents=True, exist_ok=True)
print("Arquivos em:", WORK)


## 4. Biblioteca visual com origem comprovada (opcional)

Se quiser imagens ou vídeos reais, envie arquivos para `/content/moneyPrintTest/assets/` e adicione os metadados abaixo. O sistema só utiliza mídia marcada como verificada, com licença e atribuição.

O Studio V3 escolhe automaticamente os arquivos pertinentes ao texto. Na ausência de mídia aprovada, cria slides e motions existentes, sem inventar capturas de tela.

Exemplo de item (não é ativado automaticamente):
```python
{"id":"demo-pdv","path":"assets/demo_pdv.mp4","verified":True,
 "license":"licença de uso autorizada","attribution":"material próprio",
 "title":"Demonstração de PDV","tags":["pdv","restaurante","atendimento"]}```


In [ ]:
APPROVED_ASSETS = []  # inclua somente arquivos aprovados e existentes em REPO/assets/
from scripts.colab_v4 import create_board
board = create_board(TOPIC, SCRIPT, aspect=ASPECT,
                     visual_style=VISUAL_STYLE, approved_assets=APPROVED_ASSETS)
(WORK / "storyboard_inicial.json").write_text(
    json.dumps(board, ensure_ascii=False, indent=2), encoding="utf-8")
print(len(board["scenes"]), "cenas planejadas; storyboard:", WORK / "storyboard_inicial.json")


## 5. Gerar voz real PT-BR e sincronizar cenas

**Paradee:** utiliza o mesmo backend ONNX do MoneyPrint, gerando áudio por cena com cache SHA256 e remoção conservadora de pausas. Não troca voz silenciosamente.

**Qwen3:** ativar apenas quando desejar comparar vozes; modelo mais pesado e referência autorizada. A execução falha explicitamente se a GPU não estiver disponível.

Nenhuma cena é esticada para acompanhar o áudio: o storyboard recebe os tempos reais da locução.


In [ ]:
if VOICE_ENGINE == "qwen3":
    !pip install -q 'qwen-tts'
from scripts.colab_v4 import create_narration
begin_voice = time.perf_counter()
aligned, narration, speech_report = create_narration(
    board, WORK, engine=VOICE_ENGINE, speed=VOICE_SPEED,
    reference_audio=QWEN_REFERENCE_AUDIO or None,
    reference_text=QWEN_REFERENCE_TEXT or None)
print(f"Narração pronta em {time.perf_counter()-begin_voice:.1f}s:",
      narration, "| duração:",
      round(aligned["total_frames"]/aligned["fps"], 2), "s")
from IPython.display import Audio, display
display(Audio(filename=str(narration)))


## 6. Renderizar somente as cenas alteradas

Usa o **renderizador FFmpeg Studio V3** do próprio projeto e seus motions. O cache inclui o conteúdo da cena, o arquivo visual aprovado e a versão do renderizador.

Na segunda execução, cenas aprovadas e inalteradas são reaproveitadas. O áudio final é muxado somente uma vez para não criar lacunas entre clips.

**Dica:** o notebook pode continuar em CPU mesmo com T4. GPU não acelera automaticamente o filtro FFmpeg/React.


In [ ]:
from scripts.colab_v4 import render_incremental
begin_render = time.perf_counter()
render_result = render_incremental(
    aligned, narration, REPO, WORK / "output",
    cache_dir=WORK / "visual_cache")
movie = Path(render_result["output"])
print("Tempo de render:", round(time.perf_counter()-begin_render, 1), "s")
print("Cenas novas:", render_result["scenes_rendered"])
print("Cenas reaproveitadas:", render_result["scenes_reused"])
print("MP4:", movie)


## 7. QA audiovisual e alinhamento observado por palavras

Confere dimensões, existência de áudio, total de frames, diferença de duração e silêncios prolongados. O Whisper local produz timestamps de palavras observados quando selecionado.

**Limite:** validação técnica não substitui revisão humana de naturalidade, tipografia, ritmo ou enquadramento.


In [ ]:
from scripts.colab_v4 import audiovisual_qa, transcribe_words
qa = audiovisual_qa(movie, aligned, WORK / "output")
print(json.dumps(qa, ensure_ascii=False, indent=2))
if not qa["pass"]:
    print("ATENÇÃO: QA apontou falhas; revise antes de publicar.")
transcript = None
if GENERATE_WORD_TIMESTAMPS:
    transcript = transcribe_words(movie, WORK / "output", model_size="small")
    print("Segmentos transcritos:", len(transcript))


## 8. Shorts 9:16 (opcional)

Seleciona trechos não sobrepostos com fala contínua, converte para 1080×1920, preserva a tela original dentro do quadro e usa fundo desfocado. **Não publica automaticamente**.


In [ ]:
if GENERATE_SHORTS:
    from scripts.colab_v4 import export_generic_shorts
    if transcript is None:
        transcript = transcribe_words(movie, WORK / "output")
    shorts = export_generic_shorts(movie, WORK / "shorts", transcript,
                                  count=SHORTS_COUNT)
    print(json.dumps(shorts, ensure_ascii=False, indent=2))
else:
    print("Geração de Shorts desativada.")


## 9. Motor Remotion (opcional, fora do core open source)

**Remotion não é open source OSI e sua licença pode exigir pagamento dependendo do uso.** Só execute esta célula após verificar elegibilidade e autorizar a instalação. Ela não é necessária para o funcionamento do MoneyPrint.

Este modo renderiza o projeto completo e substitui a faixa de áudio pela mesma narração Paradee/Qwen aprovada. O cache incremental padrão continua sendo do FFmpeg.


In [ ]:
ENABLE_REMOTION_EXPERIMENT = False
if ENABLE_REMOTION_EXPERIMENT:
    import copy
    optional = copy.deepcopy(aligned)
    optional["render_engine"] = "remotion"
    remotion_root = REPO / "motion" / "remotion"
    subprocess.run(["npm", "install", "--no-audit", "--no-fund"],
                   cwd=remotion_root, check=True)
    subprocess.run(["npx", "remotion", "browser", "ensure"],
                   cwd=remotion_root, check=True)
    from scripts.remotion_render import render_project as render_remotion
    remotion = render_remotion(optional, REPO, WORK / "remotion",
                              voice="none")
    remotion_out = WORK / "remotion" / "MoneyPrint_V4_Remotion_voz_ptbr.mp4"
    subprocess.run(["ffmpeg", "-hide_banner", "-loglevel", "error", "-y",
                    "-i", remotion["output"], "-i", str(narration),
                    "-map", "0:v:0", "-map", "1:a:0", "-c:v", "copy",
                    "-c:a", "aac", "-b:a", "192k", "-shortest",
                    str(remotion_out)], check=True)
    print("Prévia experimental:", remotion_out)
else:
    print("Remotion não instalado; core open source preservado.")


## 10. Empacotar artefatos para revisão

Inclui MP4 final, storyboard sincronizado, narração, QA e transcrição opcional. O cache e os modelos grandes não entram no ZIP.

Revise o arquivo antes de disponibilizar no YouTube, TikTok ou redes sociais.


In [ ]:
import zipfile
bundle = WORK / "MoneyPrint_V4_Colab_Resultado.zip"
candidates = [movie, narration, WORK / "storyboard_inicial.json",
              WORK / "storyboard_sincronizado.json",
              WORK / "voz_diagnostico.json",
              WORK / "output" / "qa_audio_video.json",
              WORK / "output" / "incremental.json",
              WORK / "output" / "palavras_reais.json"]
candidates += sorted((WORK / "shorts").glob("*.mp4")) if (WORK / "shorts").exists() else []
with zipfile.ZipFile(bundle, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for p in candidates:
        if p.exists():
            z.write(p, arcname=p.relative_to(WORK))
print("Pacote:", bundle)
from google.colab import files
files.download(str(bundle))
